<a href="https://colab.research.google.com/github/Saharochek98/Statistics-of-Stochastic-Processes/blob/main/04_sde_parameter_estimation_and_statistical_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.optimize import minimize
from numba import jit, njit
from scipy.stats import  t

In [ ]:
class StochasticProcess:
    def __init__(self, params, T, Nx, Nt, init_state = None):
        self.params = params
        self.T = T
        self.Nx = Nx # Number of trajectories
        self.Nt = Nt # Number of discretization steps
        self.init_state = init_state

    # Wiener Process by default
    # use @staticmethod for numba compatibility with @njit
    @staticmethod
    @njit
    def bxt(x, t, params):
        return 0

    @staticmethod
    @njit
    def sigmaxt(x, t, params):
        return 1

    @staticmethod
    @njit
    def bxt_x(x, t, params):
        return 0

    @staticmethod
    @njit
    def bxt_xx(x, t, params):
        return 0

    @staticmethod
    @njit
    def sigmaxt_x(x, t, params):
        return 0

    @staticmethod
    @njit
    def sigmaxt_xx(x, t, params):
        return 0

    @staticmethod
    @njit
    def bxt_t(x, t, params):
        return 0

    def DefaultInitState(self):
        return np.zeros(self.Nx)

    def EulerSolution(self, dwt = None):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if dwt is None:
            dwt = np.random.normal(0, 1, size = (self.Nt, self.Nx)) * np.sqrt(dt)

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] + self.bxt(x_data[i - 1], t_data[i - 1], self.params) * dt +\
                        self.sigmaxt(x_data[i - 1], t_data[i - 1], self.params) * dwt[i - 1]
        return t_data, x_data

    @staticmethod
    @njit
    def EulerDensity(x, t, x0, t0, params, bxt, bxt_x, bxt_xx, sigmaxt, sigmaxt_x, sigmaxt_xx, bxt_t):
        xs = x0 + bxt(x0, t0, params) * (t - t0)
        sigma2 = sigmaxt(x0, t0, params)**2 * (t - t0)

        if sigma2 <= 0 or not np.isfinite(sigma2):
            return -1e12
        return -0.5 * np.log(2 * np.pi * sigma2) - (x - xs)**2 / (2 * sigma2)

    @staticmethod
    @njit
    def OzakiDensity(x, t, x0, t0, params, bxt, bxt_x, bxt_xx, sigmaxt, sigmaxt_x, sigmaxt_xx, bxt_t):
        dt = t - t0
        L = bxt_x(x0, t0, params)
        b = bxt(x0, t0, params)
        vol2 = sigmaxt(x0, t0, params)**2
        if abs(L) < 1e-10:
            mean = x0 + b * dt
            var = vol2 * dt
        else:
            mean = x0 + b * np.expm1(L * dt) / L
            var = vol2 * np.expm1(2 * L * dt) / (2 * L)
        if var <= 0 or not np.isfinite(var):
            return -1e12
        return -0.5 * np.log(2 * np.pi * var) - (x - mean)**2 / (2 * var)

    @staticmethod
    @njit
    def ShojiOzakiDensity(x, t, x0, t0, params, bxt, bxt_x, bxt_xx, sigmaxt, sigmaxt_x, sigmaxt_xx, bxt_t):
        dt = t - t0
        L = bxt_x(x0, t0, params)
        b = bxt(x0, t0, params)
        M = 0.5 * sigmaxt(x0, t0, params)**2 * bxt_xx(x0, t0, params) + bxt_t(x0, t0, params)
        vol2 = sigmaxt(x0, t0, params)**2
        if abs(L) < 1e-10:
            mean = x0 + b * dt + 0.5 * M * dt**2
            var = vol2 * dt
        else:
            mean = x0 + b * np.expm1(L * dt) / L + M * (np.expm1(L * dt) - L * dt) / L**2
            var = vol2 * np.expm1(2 * L * dt) / (2 * L)
        if var <= 0 or not np.isfinite(var):
            return -1e12
        return -0.5 * np.log(2 * np.pi * var) - (x - mean)**2 / (2 * var)

    @staticmethod
    @njit
    def KesslerDensity(x, t, x0, t0, params, bxt, bxt_x, bxt_xx, sigmaxt, sigmaxt_x, sigmaxt_xx, bxt_t):
        E = x0 + bxt(x0, t0, params) * (t - t0) + (bxt(x0, t0, params) * bxt_x(x0, t0, params) + 1/2 * \
            sigmaxt(x0, t0, params)**2 * bxt_xx(x0, t0, params)) * 1/2 * (t - t0)**2
        V = x0**2 + (2 * bxt(x0, t0, params) * x0 + sigmaxt(x0, t0, params)**2 ) * (t - t0)\
            + (2 * bxt(x0, t0, params) * (bxt_x(x0, t0, params) * x0 + bxt(x0, t0, params) + \
                sigmaxt(x0, t0, params) * sigmaxt_x(x0, t0, params))\
            + sigmaxt(x0, t0, params)**2 * (bxt_xx(x0, t0, params) * x0 + 2 * bxt_x(x0, t0, params) +\
                sigmaxt_x(x0, t0, params)**2 + sigmaxt(x0, t0, params) * sigmaxt_xx(x0, t0, params))) * (t - t0)**2 / 2 - E**2

        xs = E

        #safe division
        if V <= 0 or not np.isfinite(V):
            return -1e12

        sigma2 = V

        return -1/2 * np.log(2 * np.pi * sigma2) - (x - xs)**2 / (2 * sigma2)

    @staticmethod
    @njit
    def MLogLik(params, t_data, x_data, density, bxt, bxt_x, bxt_xx, sigmaxt, sigmaxt_x, sigmaxt_xx, bxt_t):
        Nt = len(x_data)
        Nx = len(x_data[0])
        log_data = np.zeros(Nx)
        for k in range(0, Nx):
            m_log_lik = 0
            for j in range(1, Nt):
                m_log_lik += density(x_data[j][k], t_data[j], x_data[j - 1][k], t_data[j - 1], params,
                                     bxt, bxt_x, bxt_xx, sigmaxt, sigmaxt_x, sigmaxt_xx, bxt_t)
            log_data[k] = m_log_lik

        res = -np.mean(log_data)
        return res

    def parameter_bounds(self):
        if isinstance(self, OUprocess):
            return [(1e-5, None), (None, None), (1e-5, None)]
        if isinstance(self, CKLSprocess):
            return [(None, None), (None, None), (1e-5, None), (0.0, 2.0)]
        if isinstance(self, Task5process):
            return [(None, None), (1e-5, None)]
        if isinstance(self, HullWhiteprocess):
            return [(1e-5, None), (None, None), (1e-5, None)]
        return None

    def EulerEstimation(self, t_data, x_data):
        x0 = np.asarray(self.params, dtype=float).copy()
        accuracy = 1e-8

        min_result = minimize(self.MLogLik,
                              x0 = x0,
                              args=(t_data, x_data, self.EulerDensity,
                              self.bxt, self.bxt_x, self.bxt_xx, self.sigmaxt, self.sigmaxt_x, self.sigmaxt_xx,
                              self.bxt_t),
                              method = 'L-BFGS-B',
                              bounds=self.parameter_bounds(),
                              options={'ftol': accuracy, 'eps': accuracy})
        return min_result

    def OzakiEstimation(self, t_data, x_data):
        x0 = np.asarray(self.params, dtype=float).copy()
        accuracy = 1e-8

        min_result = minimize(self.MLogLik,
                              x0 = x0,
                              args=(t_data, x_data, self.OzakiDensity,
                              self.bxt, self.bxt_x, self.bxt_xx, self.sigmaxt, self.sigmaxt_x, self.sigmaxt_xx,
                              self.bxt_t),
                              method = 'L-BFGS-B',
                              bounds=self.parameter_bounds(),
                              options={'ftol': accuracy, 'eps': accuracy})
        return min_result

    def ShojiOzakiEstimation(self, t_data, x_data):
        x0 = np.asarray(self.params, dtype=float).copy()
        accuracy = 1e-8

        min_result = minimize(self.MLogLik,
                              x0 = x0,
                              args=(t_data, x_data, self.ShojiOzakiDensity,
                              self.bxt, self.bxt_x, self.bxt_xx, self.sigmaxt, self.sigmaxt_x, self.sigmaxt_xx,
                              self.bxt_t),
                              method = 'L-BFGS-B',
                              bounds=self.parameter_bounds(),
                              options={'ftol': accuracy, 'eps': accuracy})
        return min_result

    def KesslerEstimation(self, t_data, x_data):
        x0 = np.asarray(self.params, dtype=float).copy()
        accuracy = 1e-8

        min_result = minimize(self.MLogLik,
                              x0 = x0,
                              args=(t_data, x_data, self.KesslerDensity,
                              self.bxt, self.bxt_x, self.bxt_xx, self.sigmaxt, self.sigmaxt_x, self.sigmaxt_xx,
                              self.bxt_t),
                              method = 'L-BFGS-B',
                              bounds=self.parameter_bounds(),
                              options={'ftol': accuracy, 'eps': accuracy})
        return min_result


class OUprocess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

    # params = [theta, mu, sigma]

    @staticmethod
    @njit
    def bxt(x, t, params):
        return params[0] * (params[1] - x)

    @staticmethod
    @njit
    def sigmaxt(x, t, params):
        return params[2]

    @staticmethod
    @njit
    def bxt_x(x, t, params):
        return -params[0]

    def DefaultInitState(self):
        return self.params[1] * np.ones(self.Nx)

class CKLSprocess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

    # params = [theta1, theta2, theta3]
    @staticmethod
    @njit
    def bxt(x, t, params):
        return params[0] + params[1] * x

    @staticmethod
    @njit
    def sigmaxt(x, t, params):
        return params[2] * np.maximum(x, 0.0)**params[3]

    @staticmethod
    @njit
    def bxt_x(x, t, params):
        return params[1]

    @staticmethod
    @njit
    def sigmaxt_x(x, t, params):
        return params[2] * params[3] * np.maximum(x, 1e-10)**(params[3] - 1)

    @staticmethod
    @njit
    def sigmaxt_xx(x, t, params):
        return params[2] * params[3] * (params[3] - 1) * np.maximum(x, 1e-10)**(params[3] - 2)

    def DefaultInitState(self):
        return np.full(self.Nx, 2.0)

class Task5process(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

    # params = [theta1, theta2]
    @staticmethod
    @njit
    def bxt(x, t, params):
        return params[0] * t * x

    @staticmethod
    @njit
    def sigmaxt(x, t, params):
        return params[1] * x

    @staticmethod
    @njit
    def bxt_x(x, t, params):
        return params[0] * t

    @staticmethod
    @njit
    def sigmaxt_x(x, t, params):
        return params[1]

    @staticmethod
    @njit
    def bxt_t(x, t, params):
        return params[0] * x

    def DefaultInitState(self):
        return np.full(self.Nx, 10.0)

class HullWhiteprocess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

    # params = [theta1, theta2, theta3]
    @staticmethod
    @njit
    def bxt(x, t, params):
        return params[0] * t * (params[1] * np.sqrt(t) - x)

    @staticmethod
    @njit
    def sigmaxt(x, t, params):
        return params[2] * t

    @staticmethod
    @njit
    def bxt_x(x, t, params):
        return -params[0] * t

    @staticmethod
    @njit
    def bxt_t(x, t, params):
        return params[0] * (1.5 * params[1] * np.sqrt(t) - x)

    def DefaultInitState(self):
        return self.params[1] * np.ones(self.Nx)

Task 1. Evaluate the conditional density of the Ornstein-Uhlenbeck process
$$dX_t = (3-X_t)dt+2dW_t, \quad X_0=1, \quad %\theta=(1, 3, 2), \quad
N=100000,
\quad \Delta=1$$ and calculate the
maximum likelihood estimation.  

In [ ]:
Nx = 10000  # Reduced for memory efficiency; original assignment requested 100000
Nt = 300

T = 1

params_OU = np.array([1, 3, 2])

x0 = 1
init_state = x0 * np.ones(Nx)

OUp = OUprocess(params_OU, T, Nx, Nt, init_state)

t_data, x_data_OU = OUp.EulerSolution()

In [ ]:
OUp.EulerEstimation(t_data, x_data_OU)

In [ ]:
OUp.ShojiOzakiEstimation(t_data, x_data_OU)

In [ ]:
OUp.KesslerEstimation(t_data, x_data_OU)

Task 2. Consider  the Chan-Karolyi-Longstaff-Sanders (CKLS) model
$$
 dX_{t} = (\theta_{1}+\theta_{2} X_{t}) dt + \theta_{3} X_{t}^{\theta_{4}} dW_{t},\qquad X_{0}=2
$$
with $\theta_{1}=1$, $\theta_{2}=2$, $\theta_{3}=0.5$, $\theta_{4}=0.3$.

Use the Euler method and
* generate the sample data $X_{t_i}$ with time step $\Delta t =10^{-4}$,  
* estimate drift and diffusion coefficients,
* compute confidence intervals for all parameters in a fitted SDE.  


In [ ]:
Nx = 1
Nt = 10000

T = 1

params_CKLS = np.array([1, 2, 0.5, 0.3])

x0 = 2
init_state = x0 * np.ones(Nx)

CKLSp = CKLSprocess(params_CKLS, T, Nx, Nt, init_state)

t_data, x_data_CKLS = CKLSp.EulerSolution()

In [ ]:
plt.plot(t_data, x_data_CKLS[:,0], label = 'CKLS process')

plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

In [ ]:
CKLSp.EulerEstimation(t_data, x_data_CKLS)

In [ ]:
#Confidence intervals
n = 50
result = np.zeros((n, 4))
for k in range(0, n):
    t_data, x_data_CKLS = CKLSp.EulerSolution()
    result[k] = CKLSp.EulerEstimation(t_data, x_data_CKLS).x


alpha = 0.05
q = t.ppf(1-alpha/2, n - 1)
for i in range(0, 4):
    m = np.mean(result[:,i])
    v = np.std(result[:,i], ddof=1)
    print(f"theta{i} = {m} +- {v * q / np.sqrt(n)} with {100 - alpha * 100}% prob")

Task 3. Consider the Vasicek model
$$
dX_{t} = \theta_{1} (\theta_{2}- X_{t}) dt + \theta_{3} dW_{t},\qquad X_{0}=5
$$
with $\theta_{1}=3$, $\theta_{2}=2$ and $\theta_{3}=0.5$.

Use the Ozaki method and
*   generate the sample data $X_{t_i}$, time step $\Delta t =10^{-2}$,
*   estimate drift and diffusion coefficients,
*   compute confidence intervals for all parameters in a fitted SDE.

In [ ]:
Nx = 1
Nt = 100

T = 1

params_OU = np.array([3, 2, 0.5])

x0 = 5
init_state = x0 * np.ones(Nx)

OUp = OUprocess(params_OU, T, Nx, Nt, init_state)

t_data, x_data_OU = OUp.EulerSolution()

In [ ]:
OUp.OzakiEstimation(t_data, x_data_OU)

In [ ]:
#Confidence intervals
n = 50
result = np.zeros((n, 3))
for k in range(0, n):
    t_data, x_data_OU = OUp.EulerSolution()
    result[k] = OUp.OzakiEstimation(t_data, x_data_OU).x


alpha = 0.05
q = t.ppf(1-alpha/2, n - 1)
for i in range(0, 3):
    m = np.mean(result[:,i])
    v = np.std(result[:,i], ddof=1)
    print(f"theta{i} = {m} +- {v * q / np.sqrt(n)} with {100 - alpha * 100}% prob")

Task 4. Consider the model
$$
  dX_{t} = a(t)X_{t} dt + \theta_{2}X_{t} dW_{t},\qquad X_{0}=10
$$
with $a(t) = \theta_{1}t$, $\theta_{1}=-2$, $\theta_{2}=0.2$.

Use the Shoji-Ozaki method and:
* generate the sample data $X_{t_i}$ time step $\Delta t =10^{-3}$,  
* estimate drift and diffusion coefficients,
* compute confidence intervals for all parameters in a fitted SDE.

In [ ]:
Nx = 1
Nt = 1000

T = 1

params_T5 = np.array([-2, 0.2])

x0 = 10
init_state = x0 * np.ones(Nx)

T5p = Task5process(params_T5, T, Nx, Nt, init_state)

t_data, x_data_T5 = T5p.EulerSolution()

In [ ]:
T5p.ShojiOzakiEstimation(t_data, x_data_T5)

In [ ]:
plt.plot(t_data, x_data_T5[:,0], label = 'Task 5 process')

plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

In [ ]:
#Confidence intervals
n = 50
result = np.zeros((n, 2))
for k in range(0, n):
    t_data, x_data_T5 = T5p.EulerSolution()
    result[k] = T5p.ShojiOzakiEstimation(t_data, x_data_T5).x


alpha = 0.05
q = t.ppf(1-alpha/2, n - 1)
for i in range(0, 2):
    m = np.mean(result[:,i])
    v = np.std(result[:,i], ddof=1)
    print(f"theta{i} = {m} +- {v * q / np.sqrt(n)} with {100 - alpha * 100}% prob")

Task 5. Consider the Hull-White (extended Vasicek) model
$$
  dX_{t} = a(t)(b(t)-X_{t}) dt + \sigma(t) dW_{t},\qquad X_{0}=2
$$
with $a(t) = \theta_{1}t$, $b(t)=\theta_{2}\sqrt{t}$, the volatility depends on time $\sigma(t)=\theta_{3}t$.

Generate sample data of $X_t$ with time step $\Delta t =10^{-3}$ and $\theta_{1}=3$, $\theta_{2}=1$ and $\theta_{3}=0.3$, then
use the Kessler method and:
*   estimate drift and diffusion coefficients,
*   compute confidence intervals for all parameters in a fitted SDE.

In [ ]:
Nx = 1
Nt = 100

T = 1

params_HW = np.array([3, 1, 0.3])

x0 = 2
init_state = x0 * np.ones(Nx)

HWp = HullWhiteprocess(params_HW, T, Nx, Nt, init_state)

t_data, x_data_HW = HWp.EulerSolution()

In [ ]:
plt.plot(t_data, x_data_HW[:,0], label = 'HW process')

plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

In [ ]:
HWp.KesslerEstimation(t_data, x_data_HW)

## Experiment 1 — Exact versus Euler approximate maximum likelihood
1. Simulate the Ornstein–Uhlenbeck (Vasicek) process with a chosen parameter vector $\theta=(\theta_1,\theta_2,\theta_3)$.
2. Estimate the parameters analytically from the exact Gaussian transition density and numerically by minimizing the Euler negative log-likelihood.
3. Compare absolute and relative differences between the two estimators.
4. Discuss the effects of discretization and numerical optimization.


In [ ]:
def task1():
    true_params = np.array([2.0, 5.0, 0.5])
    T = 10.0
    Nt = 1000
    Nx = 1  # Use the same single path for exact and Euler likelihoods

    ou_process = OUprocess(params=true_params, T=T, Nx=Nx, Nt=Nt)
    t_data, x_data = ou_process.EulerSolution()

    dt = T / Nt
    x = x_data[:, 0]
    x_curr = x[:-1]
    x_next = x[1:]

    mean_curr = np.mean(x_curr)
    mean_next = np.mean(x_next)
    cov_xx = np.mean((x_curr - mean_curr) * (x_next - mean_next))
    var_curr = np.mean((x_curr - mean_curr)**2)

    phi_hat = cov_xx / var_curr
    c_hat = mean_next - phi_hat * mean_curr
    residuals = x_next - (c_hat + phi_hat * x_curr)
    sigma_eps_sq_hat = np.mean(residuals**2)

    theta1_analytic = -np.log(max(phi_hat, 1e-6)) / dt
    theta2_analytic = c_hat / (1 - phi_hat) if abs(1 - phi_hat) > 1e-6 else mean_curr
    exp_term = np.exp(-2 * theta1_analytic * dt)
    theta3_analytic = np.sqrt(sigma_eps_sq_hat * 2 * theta1_analytic / (1 - exp_term)) if (1 - exp_term) > 1e-6 else np.sqrt(sigma_eps_sq_hat / dt)

    params_analytic = np.array([theta1_analytic, theta2_analytic, theta3_analytic])

    theta1_init = 1.0
    theta2_init = 1.0
    theta3_init = 1.0

    x0_num = np.array([theta1_init, theta2_init, theta3_init])

    bounds = [(0.01, 10.0), (-5.0, 10.0), (0.01, 5.0)]

    min_result_bounded = minimize(
        ou_process.MLogLik,
        x0=x0_num,
        args=(t_data, x_data, ou_process.EulerDensity,
              ou_process.bxt, ou_process.bxt_x, ou_process.bxt_xx,
              ou_process.sigmaxt, ou_process.sigmaxt_x, ou_process.sigmaxt_xx,
              ou_process.bxt_t),
        method='L-BFGS-B',
        bounds=bounds,
        options={'ftol': 1e-10, 'gtol': 1e-8, 'maxiter': 1000}
        )

    if not min_result_bounded.success:
        print("Warning: Euler MLE optimization did not converge:", min_result_bounded.message)
    params_numerical = min_result_bounded.x
    rel_error = np.abs(params_numerical - params_analytic) / np.abs(params_analytic)

    print('Parameter     True      Exact MLE    Euler MLE    Relative difference')
    names = ['theta1', 'theta2', 'theta3']
    for i in range(3):
        print(f"{names[i]}       {true_params[i]:.4f}      {params_analytic[i]:.4f}      {params_numerical[i]:.4f}      {rel_error[i]:.2%}")

task1()

**Interpretation.**
The Exact Maximum Likelihood Estimator (MLE) and Euler-based MLE produce closely aligned parameter estimates, with relative differences between the two methods ranging from 0.00% to 1.37%.

Both methods accurately recover θ₂, while θ₃ is estimated with relatively small deviations from its true value. However, both estimators substantially overestimate θ₁, suggesting that this parameter is more difficult to recover from the observed sample.

Although Exact MLE uses the exact transition density, it does not necessarily outperform Euler MLE for every parameter in a single finite-sample experiment.

**Overall**, the results demonstrate strong agreement between the two estimation methods, but also highlight the distinction between agreement across estimators and accuracy relative to the true parameter values.

## Experiment 2 — CKLS parameter estimation
1. Simulate the Chan–Karolyi–Longstaff–Sanders (CKLS) model with a small time step $\Delta t=10^{-4}$.
2. Estimate $\theta=(\theta_1,\theta_2,\theta_3,\theta_4)$ using Euler Gaussian pseudo-likelihood.
3. Repeat the experiment 50 times and construct 95% Student-t confidence intervals **for the Monte Carlo mean of each estimate**.
4. Assess numerical stability, optimization success, and possible parameter identifiability issues.


In [ ]:
def EulerLogLik(params, t_data, x_data, process):
    Nt = len(t_data)
    Nx = len(x_data[0])

    total_loglik = np.zeros(Nx)

    for k in range(Nx):
        ll = 0.0

        for j in range(1, Nt):
            x_prev = x_data[j-1][k]
            x_curr = x_data[j][k]
            dt = t_data[j] - t_data[j-1]

            mu = process.bxt(x_prev, t_data[j-1], params)
            sigma = process.sigmaxt(x_prev, t_data[j-1], params)

            mean = x_prev + mu * dt
            var = sigma**2 * dt

            if var <= 1e-12:
                var = 1e-12

            ll += -0.5 * np.log(2*np.pi*var) - (x_curr - mean)**2 / (2*var)

        total_loglik[k] = ll

    return -np.mean(total_loglik)

def task2():
    np.random.seed(42)
    true_params = np.array([0.5, -1.0, 0.3, 0.7])

    T = 1.0
    dt = 1e-3  # Computationally reduced from requested 1e-4; see notes
    Nt = int(T / dt)
    Nx = 1

    n_sim = 50
    estimates = np.zeros((n_sim, 4))
    bounds = [
        (-2.0, 2.0),
        (-2.0, 2.0),
        (0.01, 2.0),
        (0.1, 2.0)
    ]
    for i in range(n_sim):
        process = CKLSprocess(
            params=true_params,
            T=T,
            Nx=Nx,
            Nt=Nt,
            init_state=np.array([2.0])
        )
        t_data, x_data = process.EulerSolution()
        x0 = true_params + 0.1 * np.random.randn(4)
        res = minimize(
            EulerLogLik,
            x0=x0,
            args=(t_data, x_data, process),
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 500, 'ftol': 1e-8}
        )
        estimates[i] = res.x if res.success else np.nan
    if np.isnan(estimates).any():
        print(f"Warning: {np.isnan(estimates[:, 0]).sum()} optimization runs failed.")
    estimates = estimates[~np.isnan(estimates).any(axis=1)]
    if len(estimates) < 2:
        raise RuntimeError("Insufficient successful fits for confidence intervals")
    mean_est = np.mean(estimates, axis=0)
    std_est = np.std(estimates, axis=0, ddof=1)

    t_val = t.ppf(0.975, df=len(estimates) - 1)
    ci_low = mean_est - t_val * std_est / np.sqrt(len(estimates))
    ci_high = mean_est + t_val * std_est / np.sqrt(len(estimates))

    names = ['theta1', 'theta2', 'theta3', 'theta4']
    print("Parameter   True    Mean estimate    95% CI lower    95% CI upper")

    for i in range(4):
        print(
            f"{names[i]:6} "
            f"{true_params[i]:.3f}   "
            f"{mean_est[i]:.3f}   "
            f"{ci_low[i]:.3f}   "
            f"{ci_high[i]:.3f}"
        )

task2()

**Interpretation**

The CKLS parameter estimation experiment shows different levels of accuracy across the four model parameters.

The diffusion-related parameters θ₃ and θ₄ are estimated accurately, with mean estimates of 0.299 and 0.713 compared with their true values of 0.300 and 0.700, respectively. Both true values lie within the reported 95% confidence intervals.

In contrast, the drift-related parameters θ₁ and θ₂ exhibit substantial estimation errors. Their mean estimates are 0.947 and −1.327, compared with true values of 0.500 and −1.000. Neither true value lies within its corresponding confidence interval.

These results suggest that the drift parameters are more difficult to estimate reliably under the selected simulation and estimation settings.

**Overall**, the experiment demonstrates accurate recovery of the diffusion parameters but notable inaccuracies in the drift parameters. Further investigation of sample size, parameter identifiability, optimization stability, and possible estimator bias is recommended.


## Experiment 3 — Local linearization (Ozaki method)
1. Estimate Vasicek parameters using the Ozaki approximation.
2. Compare Ozaki and Euler estimators for $\Delta t\in\{10^{-1},10^{-2},10^{-3}\}$.
3. Investigate whether the exact local linearization improves approximation accuracy at larger time steps.
4. Interpret the results, distinguishing approximation bias from Monte Carlo variation.


In [ ]:
theta_true = 2.0
mu_true = 5.0
sigma_true = 0.5

def simulate_OU(theta, mu, sigma, T, dt, Nx):
    N = int(T / dt)
    X = np.zeros((Nx, N + 1))

    for k in range(Nx):
        x = mu
        X[k, 0] = x

        for t in range(1, N + 1):
            dw = np.sqrt(dt) * np.random.randn()
            x = x + theta * (mu - x) * dt + sigma * dw
            X[k, t] = x

    return X

def euler_ll(params, X, dt):
    theta, mu, sigma = params
    ll = 0.0

    for path in X:
        for t in range(1, len(path)):
            mean = path[t-1] + theta * (mu - path[t-1]) * dt
            var = sigma**2 * dt

            ll += -0.5*np.log(2*np.pi*var) - (path[t] - mean)**2/(2*var)

    return -ll / len(X)

def ozaki_ll(params, X, dt):
    theta, mu, sigma = params
    ll = 0.0

    exp_th = np.exp(-theta * dt)

    for path in X:
        for t in range(1, len(path)):
            x_prev = path[t-1]

            mean = mu + (x_prev - mu) * exp_th
            var = (sigma**2 / (2*theta)) * (1 - exp_th**2)

            ll += -0.5*np.log(2*np.pi*var) - (path[t] - mean)**2/(2*var)

    return -ll / len(X)

def estimate(X, dt, ll_func):
    x0 = np.array([1.0, 1.0, 1.0])

    res = minimize(
        ll_func,
        x0,
        args=(X, dt),
        method="L-BFGS-B",
        bounds=[(0.01, 10), (0, 10), (0.01, 5)],
        options={"maxiter": 300}
    )

    return res.x

def task3():

    np.random.seed(42)
    T = 1.0
    Nx = 50
    dts = [0.1, 0.01, 0.001]

    for dt in dts:

        print("Δt =", dt)

        X = simulate_OU(theta_true, mu_true, sigma_true, T, dt, Nx)

        euler_est = estimate(X, dt, euler_ll)
        ozaki_est = estimate(X, dt, ozaki_ll)

        true = np.array([theta_true, mu_true, sigma_true])

        print("True  :", np.round(true, 4))
        print("Euler :", np.round(euler_est, 4))
        print("Ozaki :", np.round(ozaki_est, 4))

        print("Euler absolute error:", np.round(np.abs(euler_est - true), 4))
        print("Ozaki absolute error:", np.round(np.abs(ozaki_est - true), 4))


task3()

In [ ]:
kappa = 2.0
theta = 5.0
sigma = 0.5
x0 = 2.0

dt_values = np.array([0.1, 0.01, 0.001])

rows = []

for dt in dt_values:
    # Exact transition moments (also Ozaki for Vasicek)
    exact_mean = theta + (x0 - theta) * np.exp(-kappa * dt)
    exact_var = (
        sigma**2 * -np.expm1(-2 * kappa * dt)
        / (2 * kappa)
    )

    # Euler transition moments
    euler_mean = x0 + kappa * (theta - x0) * dt
    euler_var = sigma**2 * dt

    mean_error = abs(euler_mean - exact_mean)
    var_error = abs(euler_var - exact_var)

    rows.append((dt, mean_error, var_error))

print("dt       Euler mean error    Euler variance error")
for dt, mean_err, var_err in rows:
    print(f"{dt:<8g} {mean_err:<19.8f} {var_err:.8f}")

fig, ax = plt.subplots(figsize=(8, 5))

ax.loglog(
    dt_values,
    [row[1] for row in rows],
    "o-",
    label="Euler mean approximation error"
)

ax.loglog(
    dt_values,
    [row[2] for row in rows],
    "s-",
    label="Euler variance approximation error"
)

ax.set_xlabel("Time step Δt")
ax.set_ylabel("Absolute transition-moment error")
ax.set_title("Euler Approximation Error for Vasicek Transitions")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


**Interpretation.** For the OU/Vasicek model, the Ozaki transition approximation coincides with the exact Gaussian transition law because the drift is linear and volatility is constant. Euler uses a first-order approximation. At small observation intervals the two estimates may be similar, while finite-sample variability can obscure their differences. A fixed observation horizon does not by itself guarantee consistent drift estimation as $\Delta t\to0$.


## Experiment 4 — Time-dependent model estimation (Kessler method)
1. Consider the extended Vasicek (Hull–White) model with time-dependent coefficients.
2. Estimate drift and diffusion parameters using a Kessler-type Gaussian approximation for observations generated with parameters $(3.0,1.0,0.3)$ and $\Delta t=10^{-3}$.
3. Estimate the covariance matrix of the parameter estimates across independent runs.
4. Discuss identifiability and sensitivity to the observation horizon.


In [ ]:
TRUE_PARAMS = np.array([3.0, 1.0, 0.3])
DT = 1e-3
N_SIM = 50
HORIZONS = [1.0, 5.0, 10.0]

def seasonal_mean(t, mu):
    return mu + 0.2 * np.sin(2 * np.pi * t)

def seasonal_volatility(t, sigma):
    return sigma * (1.0 + 0.2 * np.cos(2 * np.pi * t))

def drift(t, x, params):
    kappa, mu, sigma = params
    return kappa * (seasonal_mean(t, mu) - x)

def diffusion(t, params):
    kappa, mu, sigma = params
    return seasonal_volatility(t, sigma)

def simulate_hull_white(params, T, dt, rng):
    n_steps = int(round(T / dt))
    times = np.linspace(0, T, n_steps + 1)
    h = np.diff(times)

    x = np.empty(n_steps + 1)
    x[0] = params[1]

    for i in range(n_steps):
        t = times[i]
        dW = np.sqrt(h[i]) * rng.standard_normal()

        x[i + 1] = (
            x[i]
            + drift(t, x[i], params) * h[i]
            + diffusion(t, params) * dW
        )

    return times, x

def kessler_negative_loglik(params, times, x):
    kappa, mu, sigma = params

    if kappa <= 0 or sigma <= 0:
        return 1e20

    t = times[:-1]
    h = np.diff(times)
    x_prev = x[:-1]
    x_next = x[1:]

    b = drift(t, x_prev, params)
    b_x = -kappa
    b_t = kappa * 0.4 * np.pi * np.cos(2 * np.pi * t)

    vol = diffusion(t, params)
    vol_t = (
        -0.4 * np.pi * sigma * np.sin(2 * np.pi * t)
    )

    a = vol**2
    a_t = 2.0 * vol * vol_t

    conditional_mean = (
        x_prev
        + b * h
        + 0.5 * (b_t + b_x * b) * h**2
    )

    conditional_var = (
        a * h
        + 0.5 * (a_t + 2.0 * b_x * a) * h**2
    )

    if np.any(conditional_var <= 0):
        return 1e20

    residual = x_next - conditional_mean

    nll = 0.5 * np.sum(
        np.log(2 * np.pi * conditional_var)
        + residual**2 / conditional_var
    )

    return float(nll)

def estimate_parameters(times, x):
    result = minimize(
        kessler_negative_loglik,
        x0=np.array([2.0, 1.0, 0.25]),
        args=(times, x),
        method="L-BFGS-B",
        bounds=[
            (0.05, 10.0),  # kappa
            (-2.0, 5.0),   # mu
            (0.01, 2.0),   # sigma
        ],
        options={"maxiter": 500, "ftol": 1e-9}
    )

    if not result.success:
        return None

    return result.x

def run_experiment():
    rng = np.random.default_rng(42)
    names = ["kappa", "mu", "sigma"]

    summary_rows = []
    covariance_matrices = {}

    for T in HORIZONS:
        estimates = []

        for _ in range(N_SIM):
            times, x = simulate_hull_white(
                TRUE_PARAMS, T, DT, rng
            )

            estimate = estimate_parameters(times, x)

            if estimate is not None:
                estimates.append(estimate)

        estimates = np.asarray(estimates)

        if len(estimates) < 2:
            raise RuntimeError(
                f"Insufficient successful estimates for T={T}"
            )

        mean_est = estimates.mean(axis=0)
        std_est = estimates.std(axis=0, ddof=1)
        bias = mean_est - TRUE_PARAMS
        rmse = np.sqrt(
            np.mean((estimates - TRUE_PARAMS)**2, axis=0)
        )

        covariance_matrices[T] = np.cov(
            estimates, rowvar=False
        )

        print(
            f"T={T:g}: "
            f"{len(estimates)}/{N_SIM} successful fits"
        )

        for j, name in enumerate(names):
            summary_rows.append({
                "Horizon T": T,
                "Parameter": name,
                "True": TRUE_PARAMS[j],
                "Mean estimate": mean_est[j],
                "Bias": bias[j],
                "Std. dev.": std_est[j],
                "RMSE": rmse[j],
            })

    summary = pd.DataFrame(summary_rows)

    print("\nParameter estimation summary:")
    print(summary.round(4).to_string(index=False))

    for T, covariance in covariance_matrices.items():
        print(f"\nCovariance matrix, T={T:g}:")
        print(
            pd.DataFrame(
                covariance,
                index=names,
                columns=names
            ).round(6)
        )

    fig, ax = plt.subplots(figsize=(8, 5))

    for name in names:
        subset = summary[summary["Parameter"] == name]

        ax.plot(
            subset["Horizon T"],
            subset["RMSE"],
            "o-",
            linewidth=2,
            label=name
        )

    ax.set_xlabel("Observation horizon T")
    ax.set_ylabel("RMSE of parameter estimates")
    ax.set_title(
        "Hull–White Parameter Estimation: "
        "Sensitivity to Observation Horizon"
    )
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

    return summary, covariance_matrices

summary, covariance_matrices = run_experiment()


**Interpretation.**

The experiment investigates parameter estimation for a time-dependent Hull–White model using a Kessler-type Gaussian approximation with a fixed observation interval of Δt = 0.001.

The estimation accuracy is evaluated across observation horizons T = 1, 5, and 10 using independent Monte Carlo simulations.

The results show that the RMSE of the mean-reversion parameter κ decreases substantially as the observation horizon increases, from approximately 2.96 at T = 1 to 0.55 at T = 10.

The mean-level parameter μ also exhibits improved estimation accuracy over longer horizons, while the volatility parameter σ maintains a comparatively low estimation error across all three settings.

These findings indicate that the mean-reversion speed is the most difficult parameter to identify accurately from short observation periods. Extending the observation horizon provides additional information about the drift dynamics and substantially improves its estimation.

**Overall**, the results support the expected relationship between observation horizon and parameter identifiability: longer observation periods are particularly beneficial for estimating drift parameters, while diffusion parameters can be estimated relatively accurately from high-frequency observations.

## Experiment 5 — Observation schemes
1. Compare estimation under two designs: **large-sample asymptotics**, with fixed $\Delta$ and increasing $n$ ($T\to\infty$), and **high-frequency asymptotics**, with fixed $T$ and decreasing $\Delta$.
2. Compare empirical bias and variance for drift and diffusion parameters and illustrate the results graphically.
3. Explain which parameters are expected to benefit from each sampling design.


In [ ]:
TRUE = np.array([2.0, 5.0, 0.5])

class OU:
    def __init__(self, theta):
        self.kappa, self.mu, self.sigma = theta

    def simulate(self, x0, T, n):
        dt = T / n
        x = np.zeros(n + 1)
        x[0] = x0

        for i in range(1, n + 1):
            dW = np.sqrt(dt) * np.random.randn()
            x[i] = x[i-1] + self.kappa*(self.mu - x[i-1])*dt + self.sigma*dW

        return x


def mle_euler(x, dt):
    x0, x1 = x[:-1], x[1:]

    def loss(p):
        kappa, mu, sigma = p

        if sigma <= 0:
            return 1e10

        mean = x0 + kappa*(mu - x0)*dt
        var = sigma**2 * dt

        return np.sum((x1 - mean)**2 / var + np.log(var))

    res = minimize(
        loss,
        x0=np.array([1.0, 1.0, 1.0]),
        bounds=[(0.001, 10), (-10, 10), (0.001, 10)],
        method="L-BFGS-B"
    )

    return res.x

def metrics(estimates):
    estimates = np.array(estimates)
    mean = estimates.mean(axis=0)
    bias = mean - TRUE
    var = estimates.var(axis=0)
    return mean, bias, var

def large_sample():
    dt = 0.01
    x0 = 5
    reps = 50

    Ns = [50, 100, 200, 400, 800]

    results = []

    for n in Ns:
        est = []

        for _ in range(reps):
            model = OU(TRUE)
            x = model.simulate(x0, T=n*dt, n=n)
            est.append(mle_euler(x, dt))

        mean, bias, var = metrics(est)
        results.append((n, mean, bias, var))

    return results

def high_frequency():
    T = 5
    x0 = 5
    reps = 50

    dts = [0.1, 0.05, 0.02, 0.01, 0.005, 0.002]

    results = []

    for dt in dts:
        n = int(T / dt)
        est = []

        for _ in range(reps):
            model = OU(TRUE)
            x = model.simulate(x0, T, n)
            est.append(mle_euler(x, dt))

        mean, bias, var = metrics(est)
        results.append((dt, mean, bias, var))

    return results

large = large_sample()
hf = high_frequency()

print("LARGE SAMPLE")
for n, mean, bias, var in large:
    print(f"n={n}")
    print("mean:", mean)
    print("bias:", bias)
    print("var :", var)

print("HIGH FREQUENCY")
for dt, mean, bias, var in hf:
    print(f"dt={dt}")
    print("mean:", mean)
    print("bias:", bias)
    print("var :", var)

plt.figure()
plt.title("Large sample: bias norm vs n")
plt.plot([r[0] for r in large], [np.linalg.norm(r[2]) for r in large])
plt.xlabel("n")
plt.ylabel("bias")
plt.show()

plt.figure()
plt.title("High frequency: bias norm vs dt")
plt.plot([r[0] for r in hf], [np.linalg.norm(r[2]) for r in hf])
plt.gca().invert_xaxis()
plt.xlabel("dt")
plt.ylabel("bias")
plt.show()

plt.figure()
plt.title("Variance vs n (large sample)")
plt.plot([r[0] for r in large], [np.linalg.norm(r[3]) for r in large])
plt.xlabel("n")
plt.ylabel("variance")
plt.show()

**Interpretation.** Under the large-sample design, increasing the observation horizon generally improves identification of the drift parameters, although fixed-step Euler pseudo-likelihood may retain discretization bias. Under fixed-horizon high-frequency sampling, the diffusion coefficient can be estimated increasingly precisely, whereas drift uncertainty need not vanish merely because more closely spaced observations are collected. With only 50 Monte Carlo replications, the displayed bias and variance curves can be noisy; they should not be described as monotone theoretical guarantees.
